In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import FixedLocator
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import os

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['axes.unicode_minus'] = False

# ================== 输出目录 ==================
output_dir = r"E:\关中干旱小论文\研究区域图-图一"
os.makedirs(output_dir, exist_ok=True)

# ================== 基础设置 ==================
years_clim  = range(1991, 2021)
year_target = 2025

start_date = pd.to_datetime('2025-03-01')
end_date   = pd.to_datetime('2025-05-31')

# ================== 西风带范围（用于拼图统一比例） ==================
lon_min, lon_max = 80, 160
lat_min, lat_max = 10, 60

lon_ticks_sparse = [80, 100, 120, 140, 160]
lat_ticks_sparse = [10, 20, 40, 60]

# ================== 时间分段（15天，取前6段） ==================
date_ranges = []
cur = start_date
while cur <= end_date:
    pe = min(cur + timedelta(days=14), end_date)
    date_ranges.append((cur, pe))
    cur += timedelta(days=15)
date_ranges = date_ranges[:6]

# ===================== 数据路径 =====================
u_path = r"E:\uwnd\uwnd_1980_2025_merged.nc"
v_path = r"E:\vwnd\vwnd_1980_2025_merged.nc"

# =====================================================
# 底图风速距平色标：2025 - 1991–2020
# =====================================================
# 单位：m/s
# 蓝色：2025 风速偏弱；红色：2025 风速偏强
# 如实际距平过大或过小，可调整为 -15~15、-25~25 等
WSA_VMIN, WSA_VMAX = -20.0, 20.0

wsa_cmap = plt.get_cmap('RdBu_r')

wsa_norm = mcolors.TwoSlopeNorm(
    vmin=WSA_VMIN,
    vcenter=0.0,
    vmax=WSA_VMAX
)

wsa_levels = np.linspace(WSA_VMIN, WSA_VMAX, 33)

# ================== 急流轴参数（保持你定义） ==================
jet_threshold = 30
jet_lat_range = (30, 60)

# 轴线样式
JET_LW_2025 = 2.50
JET_LW_CLIM = 2.20
JET_COLOR_2025 = "#00A000"   # 绿实线
JET_COLOR_CLIM = "#D00000"   # 红虚线

# 离群剔除阈值：只剔除明显不可能的跳变
OUTLIER_JUMP_DEG = 12.0

# ================== 一次性打开并裁剪 ==================
def open_uv_all():
    du = xr.open_dataset(u_path)['uwnd']
    dv = xr.open_dataset(v_path)['vwnd']

    du = du.assign_coords(lon=((du.lon + 360) % 360)).sortby('lon')
    dv = dv.assign_coords(lon=((dv.lon + 360) % 360)).sortby('lon')

    du = du.sortby('lat')
    dv = dv.sortby('lat')

    du = du.sel(
        level=200.0,
        lon=slice(lon_min, lon_max),
        lat=slice(lat_min, lat_max)
    )
    dv = dv.sel(
        level=200.0,
        lon=slice(lon_min, lon_max),
        lat=slice(lat_min, lat_max)
    )

    du = du.sel(time=du.time.dt.month.isin([3, 4, 5]))
    dv = dv.sel(time=dv.time.dt.month.isin([3, 4, 5]))

    return du.sortby('lon').sortby('lat'), dv.sortby('lon').sortby('lat')


print("Loading uwnd/vwnd once...")
u_all, v_all = open_uv_all()

u_target = u_all.sel(time=u_all.time.dt.year == year_target)
v_target = v_all.sel(time=v_all.time.dt.year == year_target)

u_clim_all = u_all.sel(time=u_all.time.dt.year.isin(list(years_clim)))
v_clim_all = v_all.sel(time=v_all.time.dt.year.isin(list(years_clim)))

# ================== 分段平均 ==================
def mean_for_period_year(da, start, end):
    sub = da.sel(time=slice(start, end))
    if sub.time.size == 0:
        return None
    return sub.mean('time').sortby('lon').sortby('lat')


def mean_for_period_clim(da_all, start, end):
    t = da_all.time

    cond = (
        ((t.dt.month == start.month) & (t.dt.day >= start.day)) |
        ((t.dt.month >  start.month) & (t.dt.month < end.month)) |
        ((t.dt.month == end.month)   & (t.dt.day <= end.day))
    )

    sub = da_all.sel(time=cond)

    if sub.time.size == 0:
        return None

    return sub.mean('time').sortby('lon').sortby('lat')


# ================== 风速 ==================
def compute_wind_speed(u, v):
    return np.sqrt(u**2 + v**2)


# =========================================================
# 轴线算法：保持原逻辑不变
# =========================================================
def compute_jet_axis_segments(ws_da, lat_range=(30, 60), threshold=30, outlier_jump=12.0):
    ws_da = ws_da.sortby('lon').sortby('lat')
    lat = ws_da.lat.values
    lon = ws_da.lon.values
    W   = ws_da.values  # [lat, lon]

    lat_mask = (lat >= lat_range[0]) & (lat <= lat_range[1])
    if not np.any(lat_mask):
        return []

    Wsub = W[lat_mask, :]
    lat_sub = lat[lat_mask]

    if Wsub.size == 0:
        return []

    max_speed = np.nanmax(Wsub, axis=0)
    finite_col = np.any(np.isfinite(Wsub), axis=0)

    max_idx = np.full(lon.shape, -1, dtype=int)

    if np.any(finite_col):
        max_idx[finite_col] = np.nanargmax(Wsub[:, finite_col], axis=0)

    valid = (max_speed > threshold) & finite_col & (max_idx >= 0)

    if not np.any(valid):
        return []

    valid_lons = lon[valid]
    valid_lats = lat_sub[max_idx[valid]]

    keep = np.ones_like(valid_lats, dtype=bool)

    for i in range(1, len(valid_lats)):
        if abs(valid_lats[i] - valid_lats[i-1]) > outlier_jump:
            keep[i] = False

    for i in range(len(valid_lats) - 2, -1, -1):
        if keep[i] and keep[i + 1] and abs(valid_lats[i] - valid_lats[i + 1]) > outlier_jump:
            keep[i] = False

    valid_lons = valid_lons[keep]
    valid_lats = valid_lats[keep]

    if valid_lons.size == 0:
        return []

    dlon = np.nanmedian(np.diff(lon))

    if not np.isfinite(dlon) or dlon <= 0:
        dlon = 2.5

    segments = []
    start = 0

    for i in range(1, len(valid_lons)):
        if (valid_lons[i] - valid_lons[i - 1]) > (1.5 * dlon):
            segments.append((valid_lons[start:i], valid_lats[start:i]))
            start = i

    segments.append((valid_lons[start:], valid_lats[start:]))

    segments = [(x, y) for (x, y) in segments if len(x) >= 2]

    return segments



# ================== 经纬度矩形边框 ==================
def draw_domain_border(ax, lw=0.9, alpha=0.95):
    xs = [lon_min, lon_max, lon_max, lon_min, lon_min]
    ys = [lat_min, lat_min, lat_max, lat_max, lat_min]

    ax.plot(
        xs,
        ys,
        transform=ccrs.PlateCarree(),
        color='k',
        lw=lw,
        alpha=alpha,
        zorder=20
    )


# ================== 单个子图 ==================
def plot_panel(ax, uy, vy, uc, vc, title_text):
    proj = ccrs.PlateCarree()
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=proj)

    # =====================================================
    # 底图：2025 年风速距平 = 2025 风速 - 1991–2020 平均风速
    # =====================================================
    ws_y = compute_wind_speed(uy, vy).sortby('lon').sortby('lat')
    ws_c = compute_wind_speed(uc, vc).sortby('lon').sortby('lat')

    ws_anom = ws_y - ws_c
    ws_anom = ws_anom.sortby('lon').sortby('lat')

    Wcolor = np.clip(ws_anom.values, WSA_VMIN, WSA_VMAX)

    ax.contourf(
        ws_anom.lon.values,
        ws_anom.lat.values,
        Wcolor,
        levels=wsa_levels,
        cmap=wsa_cmap,
        norm=wsa_norm,
        transform=proj,
        extend='both'
    )

    # =====================================================
    # 海岸线、坐标轴、边框
    # =====================================================
    ax.add_feature(
        cfeature.COASTLINE.with_scale('110m'),
        linewidth=0.4,
        edgecolor='k',
        alpha=0.9
    )

    ax.set_xticks(lon_ticks_sparse, crs=proj)
    ax.set_yticks(lat_ticks_sparse, crs=proj)

    ax.xaxis.set_major_formatter(LongitudeFormatter(number_format='.0f'))
    ax.yaxis.set_major_formatter(LatitudeFormatter(number_format='.0f'))

    ax.tick_params(
        labelsize=8,
        length=3,
        width=0.8,
        pad=2
    )

    draw_domain_border(ax)

    # =====================================================
    # 2025 急流轴：绿实线
    # =====================================================
    segs_y = compute_jet_axis_segments(
        ws_y,
        lat_range=jet_lat_range,
        threshold=jet_threshold,
        outlier_jump=OUTLIER_JUMP_DEG
    )

    segs_y = extend_segments_to_bounds(
        segs_y,
        lon_left=lon_min,
        lon_right=lon_max,
        extend_right=True
    )

    for x, y in segs_y:
        ax.plot(
            x,
            y,
            transform=proj,
            color=JET_COLOR_2025,
            lw=JET_LW_2025,
            ls='-',
            alpha=0.98,
            zorder=30
        )

    # =====================================================
    # 1991–2020 气候态急流轴：红虚线
    # =====================================================
    segs_c = compute_jet_axis_segments(
        ws_c,
        lat_range=jet_lat_range,
        threshold=jet_threshold,
        outlier_jump=OUTLIER_JUMP_DEG
    )

    segs_c = extend_segments_to_bounds(
        segs_c,
        lon_left=lon_min,
        lon_right=lon_max,
        extend_right=True
    )

    for x, y in segs_c:
        ax.plot(
            x,
            y,
            transform=proj,
            color=JET_COLOR_CLIM,
            lw=JET_LW_CLIM,
            ls='--',
            alpha=0.95,
            zorder=29
        )

    ax.set_title(title_text, fontsize=9, pad=4)


# ================== 色标（右侧固定，不遮挡子图） ==================
def finalize_layout_with_cbar(fig, sm, cbar_label, outpath):
    plt.tight_layout(rect=[0.02, 0.02, 0.90, 0.95])

    cax = fig.add_axes([0.92, 0.18, 0.018, 0.64])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label(cbar_label)

    # 距平色标刻度
    major_ticks = np.arange(WSA_VMIN, WSA_VMAX + 0.1, 5)
    minor_ticks = np.arange(WSA_VMIN, WSA_VMAX + 0.1, 2.5)

    cbar.set_ticks(major_ticks)
    cbar.ax.yaxis.set_minor_locator(FixedLocator(minor_ticks))

    cbar.ax.tick_params(
        which='major',
        labelsize=8,
        length=4,
        width=0.8
    )

    cbar.ax.tick_params(
        which='minor',
        length=2,
        width=0.6
    )

    cbar.ax.set_ylim(WSA_VMIN, WSA_VMAX)

    plt.savefig(outpath, dpi=300, bbox_inches='tight')
    plt.show()

    print("Saved:", outpath)


# ================== 主图：6张平铺（2x3） ==================
def plot_flat(outfile):
    fig = plt.figure(figsize=(10.8, 6.6))

    axs = [
        fig.add_subplot(2, 3, i + 1, projection=ccrs.PlateCarree())
        for i in range(6)
    ]

    for i, (start, end) in enumerate(date_ranges):
        print(f"Panel {i + 1}/6: {start.strftime('%m-%d')} to {end.strftime('%m-%d')}")

        uy = mean_for_period_year(u_target, start, end)
        vy = mean_for_period_year(v_target, start, end)

        uc = mean_for_period_clim(u_clim_all, start, end)
        vc = mean_for_period_clim(v_clim_all, start, end)

        if (uy is None) or (vy is None) or (uc is None) or (vc is None):
            axs[i].set_title("Empty", fontsize=9)
            continue

        plot_panel(
            axs[i],
            uy,
            vy,
            uc,
            vc,
            f"{start:%m-%d} – {end:%m-%d}"
        )

    fig.suptitle(
        "MAM 200 hPa Wind Speed Anomaly: 2025 minus 1991–2020 mean\n"
        "Shading: wind speed anomaly; Jet axis: 2025 green solid, Clim red dashed\n"
        "Jet axis computed from absolute wind speed | Domain: 80–160E, 10–60N",
        fontsize=11,
        y=0.98
    )

    sm = ScalarMappable(norm=wsa_norm, cmap=wsa_cmap)
    sm.set_array([])

    finalize_layout_with_cbar(
        fig,
        sm,
        "Wind Speed Anomaly (m/s): 2025 - 1991–2020 mean",
        outfile
    )


# ================== 运行/输出 ==================
out = os.path.join(
    output_dir,
    "jet_flat_MAM_200hPa_windspeed_anomaly_2025_minus_1991-2020_lon80-160_lat10-60.pdf"
)

plot_flat(out)

u_all.close()
v_all.close()

print("✅ 完成：底图为2025年相对于1991–2020平均态的200 hPa风速距平；急流轴算法保持不变。")
